# Chroma CRUD Operations #

### this notebook walks through create , read , update and delete operations with a local chroma vector store ###

In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

## 1.Setup paths and the Vector Store ##

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Shivam Singh/advanced-rag-loaders/04_vector_stores')

In [4]:
# Load environment variables from the local .env file.

dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("Please add your GOOGLE_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: c:\Users\Shivam Singh\advanced-rag-loaders\04_vector_stores\.env


In [5]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: c:\Users\Shivam Singh\advanced-rag-loaders\04_vector_stores\notebooks\chroma_langchain_db


In [6]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


In [7]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [8]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [10]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [11]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des